<a href="https://colab.research.google.com/github/Nash2027/Driver_attention/blob/main/Driver_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Connection to Google Drive
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

print("Google Drive monté avec succès !")


In [ ]:
!pip install mediapipe


In [ ]:
import os, shutil, random, math
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import matplotlib.pyplot as plt
import cv2

# Paramètres
ORIGINAL = "/content/drive/MyDrive/imgs/train"   # <-- adapte si besoin
BINARY_DIR = "/content/binary_dataset"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
WARMUP_EPOCHS = 5
FINETUNE_EPOCHS = 15
FINE_TUNE_AT = 120

print("Config prête.")


In [ ]:
if os.path.exists(BINARY_DIR):
    shutil.rmtree(BINARY_DIR)
os.makedirs(os.path.join(BINARY_DIR, "attentif"), exist_ok=True)
os.makedirs(os.path.join(BINARY_DIR, "distrait"), exist_ok=True)

att_list = [os.path.join(ORIGINAL, "c0", f) for f in os.listdir(os.path.join(ORIGINAL, "c0")) if f.lower().endswith(('.jpg','.jpeg','.png'))]
dist_list = []
for i in range(1,10):
    d = os.path.join(ORIGINAL, f"c{i}")
    if os.path.isdir(d):
        dist_list += [os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith(('.jpg','.jpeg','.png'))]

print("Found", len(att_list), "attentif images")
print("Found", len(dist_list), "distrait images")


min_count = min(len(att_list), len(dist_list))
random.seed(42)
att_sel = random.sample(att_list, min_count)
dist_sel = random.sample(dist_list, min_count)


for i, p in enumerate(att_sel):
    shutil.copy(p, os.path.join(BINARY_DIR, "attentif", f"att_{i}.jpg"))
for i, p in enumerate(dist_sel):
    shutil.copy(p, os.path.join(BINARY_DIR, "distrait", f"dist_{i}.jpg"))

print("Binary dataset created:", BINARY_DIR)
print(" - attentif:", len(os.listdir(os.path.join(BINARY_DIR,"attentif"))))
print(" - distrait:", len(os.listdir(os.path.join(BINARY_DIR,"distrait"))))


In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    BINARY_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset="training",
    seed=42,
    shuffle=True
)

val_ds = keras.utils.image_dataset_from_directory(
    BINARY_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset="validation",
    seed=42,
    shuffle=True
)

print("Classes:", train_ds.class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)


In [ ]:
# use of class_weight
n_att = len(os.listdir(os.path.join(BINARY_DIR,"attentif")))
n_dis = len(os.listdir(os.path.join(BINARY_DIR,"distrait")))
total = n_att + n_dis
# weight = total / (n_classes * count)
w0 = total / (2 * n_att)
w1 = total / (2 * n_dis)
class_weights = {0: w0, 1: w1}
print("Class weights:", class_weights)


In [ ]:
# augmentation légère + preprocess_input
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10)
], name="data_augmentation")

# Inputs / pipeline functional API
img_input = keras.Input(shape=IMG_SIZE + (3,), name="img_input")
x = data_augmentation(img_input)
x = preprocess_input(x)

base_model = keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet",
    pooling="avg"
)

base_model.trainable = False  # warmup: freeze base

x = base_model(x, training=False)
x = layers.Dropout(0.4)(x)
output = layers.Dense(1, activation="sigmoid", name="out")(x)

model = keras.Model(inputs=img_input, outputs=output)

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:
from IPython.display import clear_output

class PlotLearning(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.acc = []; self.val_acc = []; self.loss = []; self.val_loss = []
    def on_epoch_end(self, epoch, logs=None):
        self.acc.append(logs.get("accuracy"))
        self.val_acc.append(logs.get("val_accuracy"))
        self.loss.append(logs.get("loss"))
        self.val_loss.append(logs.get("val_loss"))
        clear_output(wait=True)
        plt.figure(figsize=(14,5))
        plt.subplot(1,2,1); plt.plot(self.acc, label="train"); plt.plot(self.val_acc, label="val"); plt.legend(); plt.title("Accuracy")
        plt.subplot(1,2,2); plt.plot(self.loss, label="train"); plt.plot(self.val_loss, label="val"); plt.legend(); plt.title("Loss")
        plt.show()

class EpochProgress(keras.callbacks.Callback):
    def __init__(self, total_epochs): super().__init__(); self.total_epochs = total_epochs
    def on_epoch_begin(self, epoch, logs=None):
        clear_output(wait=True)
        cur = epoch+1; total=self.total_epochs
        bar_length=30
        filled = int(bar_length * (cur/total))
        bar = "█"*filled + "-"*(bar_length-filled)
        print(f"Epoch {cur}/{total}  [{bar}]  remaining: {total-cur}")

plot_cb = PlotLearning()
epoch_progress = EpochProgress(WARMUP_EPOCHS + FINETUNE_EPOCHS)
earlystop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True, verbose=1)
csv_logger = tf.keras.callbacks.CSVLogger("training_log.csv", append=True)


all_callbacks = [plot_cb, epoch_progress, earlystop, csv_logger]


In [ ]:
print("=== Phase 1 : WARMUP ===")
base_model.trainable = False  # freeze all

history_warm = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=WARMUP_EPOCHS,
    class_weight=class_weights,
    verbose=1
)


In [ ]:
# Unfreeze from FINE_TUNE_AT
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False
for layer in base_model.layers[FINE_TUNE_AT:]:
    layer.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINETUNE_EPOCHS,
    callbacks=all_callbacks,
    class_weight=class_weights,
    verbose=1
)


In [ ]:
model_path = "/content/drive/MyDrive/driver_attention_model_with_mp.keras"
model.save(model_path)
print("Modèle sauvegardé :", model_path)


Modèle sauvegardé : /content/drive/MyDrive/driver_attention_model_with_mp.keras


In [ ]:
import tensorflow as tf

model_path = "/content/drive/MyDrive/driver_attention_model_with_mp.keras"
model = tf.keras.models.load_model(model_path)

print("Modèle chargé :", model_path)


Modèle chargé : /content/drive/MyDrive/driver_attention_model_with_mp.keras


In [ ]:
# Charge the model + mediapipe setup
import mediapipe as mp
from google.colab import files
from PIL import Image
import io

model = keras.models.load_model(model_path)
mp_hands = mp.solutions.hands
mp_face = mp.solutions.face_detection

hands_detector = mp_hands.Hands(static_image_mode=True, max_num_hands=2, min_detection_confidence=0.4)
face_detector = mp_face.FaceDetection(model_selection=0, min_detection_confidence=0.4)

def mediapipe_check(img_bgr):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    face_res = face_detector.process(rgb)
    hands_res = hands_detector.process(rgb)
    has_face = face_res.detections is not None
    n_hands = 0 if hands_res.multi_hand_landmarks is None else len(hands_res.multi_hand_landmarks)
    return has_face, n_hands

def predict_with_correction(img_pil):
    # prepare image
    img = img_pil.convert("RGB").resize(IMG_SIZE)
    arr = keras.utils.img_to_array(img)
    arr_proc = preprocess_input(np.expand_dims(arr, 0))
    prob = float(model.predict(arr_proc, verbose=0)[0][0])  # prob distrait (sigmoid)
    model_label = "distrait" if prob > 0.5 else "attentif"

    # mediapipe check on original resolution (convert PIL -> BGR)
    img_cv = cv2.cvtColor(np.array(img_pil.convert("RGB")), cv2.COLOR_RGB2BGR)
    has_face, n_hands = mediapipe_check(img_cv)

    # If model says attentif but MP sees no face or less than 2 hands => force distract
    corrected_label = model_label
    corrected_prob = prob
    if model_label == "attentif" and (not has_face or n_hands < 2):
        corrected_label = "distrait (MP override)"
        # boost probability to indicate override
        corrected_prob = max(prob, 0.75)

    # return results
    return {
        "model_prob_distrait": prob,
        "model_label": model_label,
        "has_face": has_face,
        "n_hands": n_hands,
        "corrected_label": corrected_label,
        "corrected_prob": corrected_prob
    }

# Test Image
print(" Upload one or more images to test (from local PC)...")
uploaded = files.upload()
for name, data in uploaded.items():
    img = Image.open(io.BytesIO(data))
    res = predict_with_correction(img)
    display(img.resize((400,400)))
    print("Image:", name)
    print("Model prob (distrait): {:.4f}".format(res["model_prob_distrait"]))
    print("Model label:", res["model_label"])
    print("MP - face detected:", res["has_face"], "| hands count:", res["n_hands"])
    print("Corrected:", res["corrected_label"], "(prob {:.2f})".format(res["corrected_prob"]))
    print("---------------------------------------------------\n")


In [ ]:
TEST_FOLDER = "/content/binary_dataset/distrait"
import glob
results = []
for p in glob.glob(os.path.join(TEST_FOLDER, "*.jpg"))[:200]:
    img_pil = Image.open(p)
    r = predict_with_correction(img_pil)
    results.append((p, r["model_prob_distrait"], r["model_label"], r["has_face"], r["n_hands"], r["corrected_label"]))

# Quick stats
n = len(results)
n_model_distr = sum(1 for _p,prob,lab,face,hands,cor in results if prob>0.5)
n_corrected = sum(1 for _p,prob,lab,face,hands,cor in results if "override" in cor)
print(f"Tested {n} images. model predicted distract on {n_model_distr}/{n}. Corrected by MP on {n_corrected}.")


In [ ]:
# ===============================
# Confusion Matrix
# ===============================

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true = []
y_pred = []

for batch_imgs, batch_labels in val_ds:
    preds = model.predict(batch_imgs)
    preds = (preds > 0.5).astype(int)  # seuil 0.5
    y_true.extend(batch_labels.numpy().astype(int))
    y_pred.extend(preds)

# Calcul of the matrix
cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(cm, display_labels=['attentif', 'distrait'])
disp.plot(cmap='Blues')
plt.title("Matrice de confusion — Validation")
plt.show()


In [ ]:
import matplotlib.pyplot as plt



acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]

epochs = range(1, len(acc) + 1)

plt.figure(figsize=(14, 5))

# Courbe Accuracy
plt.subplot(1, 2, 1)
plt.plot(epochs, acc, label="Entraînement")
plt.plot(epochs, val_acc, label="Validation")
plt.title("Évolution de l'Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

#Courbe Loss
plt.subplot(1, 2, 2)
plt.plot(epochs, loss, label="Entraînement")
plt.plot(epochs, val_loss, label="Validation")
plt.title("Évolution de la Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.show()
